**Level 3: Numerical Methods**
**Executive Summary**

Numerical methods provide computational techniques for solving mathematical problems that cannot easily be solved analytically. In agricultural systems, numerical methods are useful for estimating irrigation requirements, modelling soil moisture changes, calculating cumulative water deficits, and optimizing resource allocation.

This notebook implements root-finding algorithms, numerical differentiation, numerical integration, and Gaussian elimination. The methods are applied to irrigation-related scenarios using the cleaned datasets. Performance, convergence behaviour, and accuracy are evaluated to determine the most suitable techniques for practical irrigation management.

**1. Introduction**

Scientific computing frequently relies on numerical approximations when exact solutions are unavailable or impractical.

In irrigation systems, numerical methods can be used to:

*   Estimate required irrigation amounts
*   Predict soil moisture trends
*   Calculate cumulative water deficits
*   Allocate water resources efficiently

This notebook explores several numerical techniques and demonstrates their application in agricultural decision support.

**2. Learning Objectives**

By the end of this notebook, you should be able to:

*   Implement root-finding algorithms manually.
*   Compare convergence behaviour of numerical methods.
*  Estimate derivatives using finite difference techniques.
* Approximate integrals using numerical integration.
* Solve systems of linear equations using Gaussian elimination.
* Interpret numerical results within irrigation management contexts.

**3. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**4.Load Dataset**

In [ ]:
weather = pd.read_csv("weather_daily_cleaned.csv")

soil = pd.read_csv("soil_sensor_data_cleaned.csv")

crop = pd.read_csv("crop_zone_parameters_cleaned.csv")

**5. Root Finding Problem
Background**

Suppose we wish to determine the irrigation amount required to reach a target soil moisture level.

The irrigation requirement can be represented by:

f(x)=x
3
−6x−5

The root of the equation corresponds to the required irrigation amount.

**Visualizing the Function**

In [ ]:
def f(x):
    return x**3 - 6*x - 5

x = np.linspace(-5,5,100)

plt.figure(figsize=(8,5))
plt.plot(x,f(x))
plt.axhline(0,color='red')
plt.grid()
plt.title("Root Finding Function")
plt.show()

**6. Bisection Method
Theory**

The Bisection Method repeatedly divides an interval into two equal halves and selects the subinterval containing the root.

Advantages:

Guaranteed convergence
Simple implementation

Disadvantages:

Slow convergence

**Implementation**

In [ ]:
def bisection(f,a,b,tol=1e-6,max_iter=100):

    iterations=[]

    for i in range(max_iter):

        c=(a+b)/2

        iterations.append(
            [i+1,c,abs(f(c))]
        )

        if abs(f(c))<tol:
            break

        if f(a)*f(c)<0:
            b=c
        else:
            a=c

    return pd.DataFrame(
        iterations,
        columns=["Iteration","Root","Error"]
    )

**Execute Method**

In [ ]:
bisection_results = bisection(f,2,3)

bisection_results.head()

**Final Estimate**

In [ ]:
bisection_results.tail(1)

**7. Newton-Raphson Method
Theory**

Newton-Raphson uses tangent lines to approximate roots.

Formula:

x
n+1
	​

=x
n
	​

−
f
′
(x
n
	​

)
f(x
n
	​

)
	​


Advantages:

* Very fast convergence

Disadvantages:

* Requires derivative
* Sensitive to initial guess

**Derivative Function**

In [ ]:
def df(x):
    return 3*x**2 - 6

**Implementation**

In [ ]:
def newton(f,df,x0,tol=1e-6,max_iter=100):

    results=[]

    x=x0

    for i in range(max_iter):

        x=x-f(x)/df(x)

        results.append(
            [i+1,x,abs(f(x))]
        )

        if abs(f(x))<tol:
            break

    return pd.DataFrame(
        results,
        columns=["Iteration","Root","Error"]
    )

**Execute Method**

In [ ]:
newton_results = newton(
    f,
    df,
    2.5
)

newton_results

**8. Secant Method**

**Theory**

The Secant Method approximates derivatives using two previous estimates.

Formula:

x
n+1
	​

=x
n
	​

−f(x
n
	​

)
f(x
n
	​

)−f(x
n−1
	​

)
x
n
	​

−x
n−1
	​

	​


Advantages:

* Faster than Bisection
* No derivative required

**Implementation**

In [ ]:
def secant(
    f,
    x0,
    x1,
    tol=1e-6,
    max_iter=100
):

    results=[]

    for i in range(max_iter):

        x2 = x1 - f(x1)*(x1-x0)/(f(x1)-f(x0))

        results.append(
            [i+1,x2,abs(f(x2))]
        )

        if abs(f(x2))<tol:
            break

        x0=x1
        x1=x2

    return pd.DataFrame(
        results,
        columns=["Iteration","Root","Error"]
    )

**Execute Method**

In [ ]:
secant_results = secant(
    f,
    2,
    3
)

secant_results

**9.Method Comparison**

In [ ]:
comparison = pd.DataFrame({

    "Method":[
        "Bisection",
        "Newton",
        "Secant"
    ],

    "Iterations":[
        len(bisection_results),
        len(newton_results),
        len(secant_results)
    ],

    "Final_Error":[
        bisection_results.iloc[-1]["Error"],
        newton_results.iloc[-1]["Error"],
        secant_results.iloc[-1]["Error"]
    ]
})

comparison

**Interpretation**

Typically:

Newton-Raphson converges fastest.
Secant converges quickly without derivatives.
Bisection is slower but highly reliable.

**10. Numerical Differentiation**

**Background**

Soil moisture change rates can be estimated using finite differences.

Consider:

f(x)=x
2

**Forward Difference**

In [ ]:
def forward_diff(f,x,h=0.01):

    return (f(x+h)-f(x))/h

**Backward Difference**

In [ ]:
def backward_diff(f,x,h=0.01):

    return (f(x)-f(x-h))/h

**Central Difference**

In [ ]:
def central_diff(f,x,h=0.01):

    return (
        f(x+h)-f(x-h)
    )/(2*h)

**Example**

In [ ]:
f2 = lambda x:x**2

x=2

print(
    forward_diff(f2,x)
)

print(
    backward_diff(f2,x)
)

print(
    central_diff(f2,x)
)

**Interpretation**

Central Difference generally produces the most accurate estimate because it uses information from both sides of the point.

**11. Numerical Integration
Background**

Water deficit accumulates over time and can be estimated through integration.

**Trapezoidal Rule**

In [ ]:
def trapezoidal(x,y):

    return np.trapz(y,x)

**Simpson's Rule**

In [ ]:
def simpson(x,y):

    h=(x[-1]-x[0])/(len(x)-1)

    return (
        h/3
    )*(
        y[0]
        + y[-1]
        + 4*sum(y[1:-1:2])
        + 2*sum(y[2:-2:2])
    )

**Example Data**

In [ ]:
x=np.array([0,1,2,3,4])

y=np.array([1,2,4,8,16])

**Execute Integration**

In [ ]:
trap_result = trapezoidal(x,y)

simp_result = simpson(x,y)

print("Trapezoidal:",trap_result)

print("Simpson:",simp_result)

**Comparison Table**

In [ ]:
integration_results = pd.DataFrame({

    "Method":[
        "Trapezoidal",
        "Simpson"
    ],

    "Integral":[
        trap_result,
        simp_result
    ]
})

integration_results

**12. Gaussian Elimination
Background**

Water allocation among irrigation zones can be modelled as:

Ax=b

where:

* A = coefficient matrix
* b = water demand vector

**Implementation**

In [ ]:
def gaussian_elimination(A,b):

    A=A.astype(float)

    b=b.astype(float)

    n=len(b)

    for i in range(n):

        for j in range(i+1,n):

            ratio=A[j,i]/A[i,i]

            A[j]-=ratio*A[i]

            b[j]-=ratio*b[i]

    x=np.zeros(n)

    for i in range(n-1,-1,-1):

        x[i]=(
            b[i]
            -
            np.dot(
                A[i,i+1:],
                x[i+1:]
            )
        )/A[i,i]

    return x

**Example System**

In [ ]:
A=np.array([
    [2,1,-1],
    [-3,-1,2],
    [-2,1,2]
])

b=np.array([
    8,
    -11,
    -3
])

**Solve System**

In [ ]:
solution = gaussian_elimination(A,b)

solution

**Interpretation**

The resulting values represent water allocations for the three irrigation zones.

These allocations can be used to distribute available water resources efficiently.

**13. Discussion**

The numerical experiments reveal important trade-offs:

**Root Finding**
* Newton-Raphson is fastest.
* Secant balances speed and simplicity.
* Bisection is most robust.
**Differentiation**
* Central Difference is most accurate.
**Integration**
* Simpson's Rule generally provides higher accuracy.
**Linear Systems**
* Gaussian Elimination efficiently solves allocation problems.

**14. Conclusion**

This notebook demonstrated the application of numerical methods to irrigation-related problems.

Key findings:

* Root-finding algorithms can determine irrigation requirements.
* Differentiation estimates moisture change rates.
* Integration estimates cumulative water deficits.
* Gaussian Elimination solves water allocation systems.